In [18]:
# FIFA World Cup 2026 Prediction Engine
## Notebook 2: Feature Engineering

# This notebook:
# - Loads the historical training dataset
# - Creates difference and ratio features
# - Removes leakage variables
# - Prepares the final modeling dataset

In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = "data/processed"

In [2]:
df = pd.read_csv(f"{PROCESSED_DIR}/training_dataset_raw.csv")

print("Shape:", df.shape)
print("\nColumn list:")
for col in df.columns:
    print(" ", col)

Shape: (384, 92)

Column list:
  home_team
  away_team
  home_score
  home_xg
  home_penalty
  away_score
  away_xg
  away_penalty
  home_manager
  home_captain
  away_manager
  away_captain
  Attendance
  Venue
  Officials
  Round
  Date
  Score
  Referee
  Notes
  Host
  Year
  home_goal
  away_goal
  home_goal_long
  away_goal_long
  home_own_goal
  away_own_goal
  home_penalty_goal
  away_penalty_goal
  home_penalty_miss_long
  away_penalty_miss_long
  home_penalty_shootout_goal_long
  away_penalty_shootout_goal_long
  home_penalty_shootout_miss_long
  away_penalty_shootout_miss_long
  home_red_card
  away_red_card
  home_yellow_red_card
  away_yellow_red_card
  home_yellow_card_long
  away_yellow_card_long
  home_substitute_in_long
  away_substitute_in_long
  year
  home_version
  home_continent
  home_is_host
  home_goals_scored_last_4y
  home_goals_received_last_4y
  home_wins_last_4y
  home_losses_last_4y
  home_draws_last_4y
  home_world_cup_titles_before
  home_squad_total_ma

In [19]:
## Leakage Removal and Feature Selection

In [3]:
# Explicit leakage columns named in the project spec
explicit_leakage = [
    "home_score",
    "away_score",
    "Score",
    "home_xg",
    "away_xg",
    "home_goal",
    "away_goal",
    "home_goal_long",
    "away_goal_long",
    "home_penalty",
    "away_penalty",
    "home_goal",
    "away_goal",
]

leakage_keywords = [
    "xg",
    "goal_long",
    "own_goal",
    "shootout",
    "penalty_goal",
    "penalty_miss",
    "red_card",
    "yellow_card",
    "substitute",
]

# Auto-detect leakage columns
auto_detected = [
    c for c in df.columns
    if any(k in c.lower() for k in leakage_keywords)
]

# Combine explicit + auto-detected
leakage_cols = sorted(
    set(
        [c for c in explicit_leakage if c in df.columns]
        + auto_detected
    )
)

print(f"Leakage columns found ({len(leakage_cols)}):")
for c in leakage_cols:
    print(" ", c)

print("\nIMPORTANT FEATURES KEPT:")
for col in [
    "home_goals_scored_last_4y",
    "away_goals_scored_last_4y",
    "home_goals_received_last_4y",
    "away_goals_received_last_4y",
]:
    if col in df.columns:
        print(" ✓", col)

Leakage columns found (29):
  Score
  away_goal
  away_goal_long
  away_own_goal
  away_penalty
  away_penalty_goal
  away_penalty_miss_long
  away_penalty_shootout_goal_long
  away_penalty_shootout_miss_long
  away_red_card
  away_score
  away_substitute_in_long
  away_xg
  away_yellow_card_long
  away_yellow_red_card
  home_goal
  home_goal_long
  home_own_goal
  home_penalty
  home_penalty_goal
  home_penalty_miss_long
  home_penalty_shootout_goal_long
  home_penalty_shootout_miss_long
  home_red_card
  home_score
  home_substitute_in_long
  home_xg
  home_yellow_card_long
  home_yellow_red_card

IMPORTANT FEATURES KEPT:
 ✓ home_goals_scored_last_4y
 ✓ away_goals_scored_last_4y
 ✓ home_goals_received_last_4y
 ✓ away_goals_received_last_4y


In [4]:
# Columns starting with home_ or away_ that are NOT leakage
home_cols = [c for c in df.columns if c.startswith("home_") and c not in leakage_cols]
away_cols = [c for c in df.columns if c.startswith("away_") and c not in leakage_cols]

# Strip the prefix to get the "base" feature name
home_bases = {c[len("home_"):] for c in home_cols}
away_bases = {c[len("away_"):] for c in away_cols}

# Bases that exist on BOTH sides - these are our candidates
# for difference/ratio features
paired_bases = sorted(home_bases & away_bases)

# Bases that only exist on one side (worth knowing about, but
# we can't build a "difference" from them)
home_only = sorted(home_bases - away_bases)
away_only = sorted(away_bases - home_bases)

print(f"Paired features ({len(paired_bases)}):")
for b in paired_bases:
    print(" ", b)

print(f"\nHome-only (no away counterpart): {home_only}")
print(f"Away-only (no home counterpart): {away_only}")

Paired features (26):
  captain
  continent
  draws_last_4y
  fifa_points_pre_tournament
  fifa_rank_pre_tournament
  finalist
  finals_before
  goals_received_last_4y
  goals_scored_last_4y
  groups_passed_before
  is_host
  losses_last_4y
  manager
  quarter_finalist
  quarterfinals_before
  round16_before
  semi_finalist
  semifinals_before
  squad_avg_age
  squad_total_market_value_eur
  team
  version
  winner
  wins_last_4y
  world_cup_participations_before
  world_cup_titles_before

Home-only (no away counterpart): []
Away-only (no home counterpart): []


In [5]:
numeric_pairs = []
non_numeric_pairs = []

for base in paired_bases:
    h_col, a_col = f"home_{base}", f"away_{base}"
    if pd.api.types.is_numeric_dtype(df[h_col]) and pd.api.types.is_numeric_dtype(df[a_col]):
        numeric_pairs.append(base)
    else:
        non_numeric_pairs.append(base)

print(f"Numeric pairs ({len(numeric_pairs)}):")
for b in numeric_pairs:
    print(" ", b)

if non_numeric_pairs:
    print(f"\nNon-numeric pairs (skipped for differences): {non_numeric_pairs}")

Numeric pairs (22):
  draws_last_4y
  fifa_points_pre_tournament
  fifa_rank_pre_tournament
  finalist
  finals_before
  goals_received_last_4y
  goals_scored_last_4y
  groups_passed_before
  is_host
  losses_last_4y
  quarter_finalist
  quarterfinals_before
  round16_before
  semi_finalist
  semifinals_before
  squad_avg_age
  squad_total_market_value_eur
  version
  winner
  wins_last_4y
  world_cup_participations_before
  world_cup_titles_before

Non-numeric pairs (skipped for differences): ['captain', 'continent', 'manager', 'team']


In [6]:
# Remove useless constant feature
if "version" in numeric_pairs:
    numeric_pairs.remove("version")

print("Updated numeric pairs:")
for b in numeric_pairs:
    print(" ", b)

print("\nCount:", len(numeric_pairs))

Updated numeric pairs:
  draws_last_4y
  fifa_points_pre_tournament
  fifa_rank_pre_tournament
  finalist
  finals_before
  goals_received_last_4y
  goals_scored_last_4y
  groups_passed_before
  is_host
  losses_last_4y
  quarter_finalist
  quarterfinals_before
  round16_before
  semi_finalist
  semifinals_before
  squad_avg_age
  squad_total_market_value_eur
  winner
  wins_last_4y
  world_cup_participations_before
  world_cup_titles_before

Count: 21


In [20]:
## Difference and Ratio Feature Engineering

In [7]:
created_diff_features = []

for base in numeric_pairs:
    h_col, a_col = f"home_{base}", f"away_{base}"
    diff_col = f"{base}_difference"

    df[diff_col] = df[h_col] - df[a_col]
    created_diff_features.append(diff_col)

print(f"Created {len(created_diff_features)} difference features:")
for c in created_diff_features:
    print(" ", c)

Created 21 difference features:
  draws_last_4y_difference
  fifa_points_pre_tournament_difference
  fifa_rank_pre_tournament_difference
  finalist_difference
  finals_before_difference
  goals_received_last_4y_difference
  goals_scored_last_4y_difference
  groups_passed_before_difference
  is_host_difference
  losses_last_4y_difference
  quarter_finalist_difference
  quarterfinals_before_difference
  round16_before_difference
  semi_finalist_difference
  semifinals_before_difference
  squad_avg_age_difference
  squad_total_market_value_eur_difference
  winner_difference
  wins_last_4y_difference
  world_cup_participations_before_difference
  world_cup_titles_before_difference


In [8]:
ratio_candidates = [
    "fifa_points_pre_tournament",
    "goals_scored_last_4y",
    "goals_received_last_4y",
    "wins_last_4y",
    "squad_avg_age",
]

# Only attempt ratios for bases that actually exist as numeric pairs
ratio_candidates = [b for b in ratio_candidates if b in numeric_pairs]

created_ratio_features = []

for base in ratio_candidates:
    h_col, a_col = f"home_{base}", f"away_{base}"
    ratio_col = f"{base}_ratio"

    # Replacing 0 with NaN in the denominator to avoid division by zero
    safe_denominator = df[a_col].replace(0, np.nan)

    df[ratio_col] = df[h_col] / safe_denominator
    created_ratio_features.append(ratio_col)

print(f"Created {len(created_ratio_features)} ratio features:")
for c in created_ratio_features:
    print(" ", c)

print("\nMissing values introduced by ratio (division by zero):")
print(df[created_ratio_features].isnull().sum())

Created 5 ratio features:
  fifa_points_pre_tournament_ratio
  goals_scored_last_4y_ratio
  goals_received_last_4y_ratio
  wins_last_4y_ratio
  squad_avg_age_ratio

Missing values introduced by ratio (division by zero):
fifa_points_pre_tournament_ratio    0
goals_scored_last_4y_ratio          0
goals_received_last_4y_ratio        0
wins_last_4y_ratio                  0
squad_avg_age_ratio                 0
dtype: int64


In [9]:
# Check for any column that might indicate the tournament host
host_related_cols = [c for c in df.columns if "host" in c.lower()]
print("Host-related columns found:", host_related_cols)

Host-related columns found: ['Host', 'home_is_host', 'away_is_host', 'is_host_difference']


In [10]:
if "Host" in df.columns:

    df["home_is_host"] = (df["home_team"] == df["Host"]).astype(int)
    df["away_is_host"] = (df["away_team"] == df["Host"]).astype(int)

    df["host_difference"] = (
        df["home_is_host"] - df["away_is_host"]
    )

    print("Created host_difference")

    print("\nDistribution:")
    print(df["host_difference"].value_counts())

else:
    print("Host column not found")

Created host_difference

Distribution:
host_difference
 0    359
 1     19
-1      6
Name: count, dtype: int64


In [11]:
# Identifier for reference
identifier_cols = ["year", "home_team", "away_team"]

# The engineered features = differences + ratios + hostdifference 
engineered_features = created_diff_features + created_ratio_features
if "host_difference" in df.columns:
    engineered_features.append("host_difference")

print(f"Total engineered features: {len(engineered_features)}\n")
for f in engineered_features:
    print(" ", f)

Total engineered features: 27

  draws_last_4y_difference
  fifa_points_pre_tournament_difference
  fifa_rank_pre_tournament_difference
  finalist_difference
  finals_before_difference
  goals_received_last_4y_difference
  goals_scored_last_4y_difference
  groups_passed_before_difference
  is_host_difference
  losses_last_4y_difference
  quarter_finalist_difference
  quarterfinals_before_difference
  round16_before_difference
  semi_finalist_difference
  semifinals_before_difference
  squad_avg_age_difference
  squad_total_market_value_eur_difference
  winner_difference
  wins_last_4y_difference
  world_cup_participations_before_difference
  world_cup_titles_before_difference
  fifa_points_pre_tournament_ratio
  goals_scored_last_4y_ratio
  goals_received_last_4y_ratio
  wins_last_4y_ratio
  squad_avg_age_ratio
  host_difference


In [12]:
print("Current target values:")
print(df["match_result"].value_counts())

expected_categories = {"Home Win", "Draw", "Away Win"}
actual_categories = set(df["match_result"].unique())

print("\nExpected categories:", expected_categories)
print("Actual categories:  ", actual_categories)
print("Match:", expected_categories == actual_categories)

Current target values:
match_result
Home Win    165
Away Win    131
Draw         88
Name: count, dtype: int64

Expected categories: {'Draw', 'Home Win', 'Away Win'}
Actual categories:   {'Draw', 'Home Win', 'Away Win'}
Match: True


In [21]:
## Final Modeling Dataset Construction

In [13]:
X = df[engineered_features].copy()
y = df["match_result"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX columns:")
print(list(X.columns))

X shape: (384, 27)
y shape: (384,)

X columns:
['draws_last_4y_difference', 'fifa_points_pre_tournament_difference', 'fifa_rank_pre_tournament_difference', 'finalist_difference', 'finals_before_difference', 'goals_received_last_4y_difference', 'goals_scored_last_4y_difference', 'groups_passed_before_difference', 'is_host_difference', 'losses_last_4y_difference', 'quarter_finalist_difference', 'quarterfinals_before_difference', 'round16_before_difference', 'semi_finalist_difference', 'semifinals_before_difference', 'squad_avg_age_difference', 'squad_total_market_value_eur_difference', 'winner_difference', 'wins_last_4y_difference', 'world_cup_participations_before_difference', 'world_cup_titles_before_difference', 'fifa_points_pre_tournament_ratio', 'goals_scored_last_4y_ratio', 'goals_received_last_4y_ratio', 'wins_last_4y_ratio', 'squad_avg_age_ratio', 'host_difference']


In [14]:
missing = X.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Missing values per feature:")
if missing.empty:
    print("  (none)")
else:
    print(missing)
    print(f"\nTotal rows in X: {len(X)}")
    print(f"Rows with at least one missing value: {X.isnull().any(axis=1).sum()}")

Missing values per feature:
squad_total_market_value_eur_difference    64
dtype: int64

Total rows in X: 384
Rows with at least one missing value: 64


In [15]:
print("Missing before:")
print(X["squad_total_market_value_eur_difference"].isna().sum())

median_mv = X["squad_total_market_value_eur_difference"].median()

X["squad_total_market_value_eur_difference"] = (
    X["squad_total_market_value_eur_difference"]
    .fillna(median_mv)
)

df["squad_total_market_value_eur_difference"] = (
    df["squad_total_market_value_eur_difference"]
    .fillna(median_mv)
)

print("\nMissing after:")
print(X["squad_total_market_value_eur_difference"].isna().sum())

Missing before:
64

Missing after:
0


In [16]:
# Final dataset = identifiers + target + engineered features)
final_columns = identifier_cols + ["match_result"] + engineered_features
training_features_df = df[final_columns].copy()

output_path = f"{PROCESSED_DIR}/training_dataset_features.csv"
training_features_df.to_csv(output_path, index=False)

print(f"Saved {training_features_df.shape[0]} rows x {training_features_df.shape[1]} columns to:")
print(f"  {output_path}")

Saved 384 rows x 31 columns to:
  data/processed/training_dataset_features.csv


In [22]:
## Summary

In [17]:
print("=" * 60)
print("NOTEBOOK 02 SUMMARY")
print("=" * 60)

print(f"\nFinal feature count: {len(engineered_features)}")
print("\nFinal feature list:")
for f in engineered_features:
    print(" ", f)

print(f"\nTarget distribution (y):")
print(y.value_counts())
print("\nTarget distribution (%):")
print(y.value_counts(normalize=True).round(3))

print(f"\nMissing value summary:")
missing = X.isnull().sum()
missing = missing[missing > 0]
print(missing if not missing.empty else "  (none)")

print(f"\nLeakage columns excluded ({len(leakage_cols)}):")
for c in leakage_cols:
    print(" ", c)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nSaved to: {PROCESSED_DIR}/training_dataset_features.csv")

NOTEBOOK 02 SUMMARY

Final feature count: 27

Final feature list:
  draws_last_4y_difference
  fifa_points_pre_tournament_difference
  fifa_rank_pre_tournament_difference
  finalist_difference
  finals_before_difference
  goals_received_last_4y_difference
  goals_scored_last_4y_difference
  groups_passed_before_difference
  is_host_difference
  losses_last_4y_difference
  quarter_finalist_difference
  quarterfinals_before_difference
  round16_before_difference
  semi_finalist_difference
  semifinals_before_difference
  squad_avg_age_difference
  squad_total_market_value_eur_difference
  winner_difference
  wins_last_4y_difference
  world_cup_participations_before_difference
  world_cup_titles_before_difference
  fifa_points_pre_tournament_ratio
  goals_scored_last_4y_ratio
  goals_received_last_4y_ratio
  wins_last_4y_ratio
  squad_avg_age_ratio
  host_difference

Target distribution (y):
match_result
Home Win    165
Away Win    131
Draw         88
Name: count, dtype: int64

Target dis